# 06 Legacy MILP-Ready Exports

This notebook packages the older exploratory Phase 5 outputs into legacy export files.

Current scope:

- export hourly actuals;
- export selected hourly forecast anchors;
- export flat and shape-adjusted 15-minute counterfactual forecast inputs;
- export counterfactual realized 15-minute paths for the required scenario variants;
- attach manifest rows and sidecar metadata for later downstream use.

Methodology note:

- this export bundle belongs to the earlier exploratory combined path;
- the frozen canonical actual path is the authoritative realized input for downstream thesis bidding work.
- any legacy Phase 6 realized-path artifacts are exploratory only and must not be treated as authorized 15-minute market truth.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
from quarterhour_da import find_latest_phase06_run

## Optional Phase 6 Runner

In [ ]:
RUN_PHASE06 = False

if RUN_PHASE06:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_phase06_milp_exports.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Phase 6 export packaging failed with exit code {completed.returncode}.")

In [ ]:
latest_run = find_latest_phase06_run(config)
if latest_run is None:
    raise FileNotFoundError("No saved Phase 6 artifact exists yet. Run the phase 6 script first.")

export_manifest = pd.read_csv(latest_run / "export_manifest.csv")
schema_checks = pd.read_csv(latest_run / "export_schema_checks.csv")
run_summary = json.loads((latest_run / "run_summary.json").read_text(encoding="utf-8"))

display(pd.DataFrame([{"latest_phase06_run": str(latest_run)}]))

## Export Manifest

In [ ]:
display(export_manifest)

## Export Schema Checks

In [ ]:
display(schema_checks)